In [1]:
import numpy as np
import pandas as pd
import joblib
import os

from scipy.stats import randint
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

from config import SEED

Elegimos dataset

In [2]:
dataset = 'aug_sin_proc'

## Espectrogramas

Se suelen usar Mel espectrogramas

In [22]:
train_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [23]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((15972, 64128), (15972,), (3996, 64128), (3996,))

### Random Forest

#### Entrenamiento

In [24]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')
param_distributions = {
    'max_depth': [15, 20, 25],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 10 , 15]
}

In [26]:
# get a random sample from the x train

indices = np.random.choice(X_train.shape[0], size=8000, replace=False)

X_train_sample = X_train[indices]
y_train_sample = y_train[indices]

In [28]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train_sample, y_train_sample)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=15, min_samples_leaf=15, min_samples_split=20;, score=0.632 total time=  50.6s
[CV 2/5] END max_depth=15, min_samples_leaf=15, min_samples_split=20;, score=0.643 total time=  58.5s
[CV 3/5] END max_depth=15, min_samples_leaf=15, min_samples_split=20;, score=0.601 total time=  59.3s


KeyboardInterrupt: 

In [10]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
10,15,5,10,0.685927
3,25,10,10,0.685656
5,25,10,20,0.685656
6,20,10,10,0.685176
1,15,10,20,0.684012
7,15,10,10,0.684012
16,20,5,20,0.683231
2,20,15,30,0.679507
0,25,15,30,0.679424
11,25,15,20,0.679424


In [11]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'min_samples_split': 10, 'min_samples_leaf': 5, 'max_depth': 15}
Best CV score: 0.6859267138539586


In [12]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1933
           1       0.99      1.00      0.99      2060

    accuracy                           0.99      3993
   macro avg       0.99      0.99      0.99      3993
weighted avg       0.99      0.99      0.99      3993



#### Evaluación

In [13]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.66      0.59      0.62       484
           1       0.65      0.71      0.68       515

    accuracy                           0.65       999
   macro avg       0.65      0.65      0.65       999
weighted avg       0.65      0.65      0.65       999



#### Guardado

In [14]:
os.makedirs(f'./modelos_clasicos/modelos/ventanas/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/ventanas/{dataset}/melspec_rf.pkl')

['./modelos_clasicos/modelos/ventanas/crudos/melspec_rf.pkl']

## Features de Audio

In [7]:
train_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [8]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((15972, 46), (15972,), (3996, 46), (3996,))

### Random Forest

#### Entrenamiento

In [14]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1, max_features=None)

param_distributions = {
    'max_depth': randint(15, 25),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(3, 10)
}

In [15]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.669 total time=  21.6s
[CV 2/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.681 total time=  22.1s
[CV 3/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.682 total time=  21.4s
[CV 4/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.696 total time=  22.3s
[CV 5/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.701 total time=  22.2s
[CV 1/5] END max_depth=22, min_samples_leaf=7, min_samples_split=19;, score=0.669 total time=  22.6s
[CV 2/5] END max_depth=22, min_samples_leaf=7, min_samples_split=19;, score=0.680 total time=  21.9s
[CV 3/5] END max_depth=22, min_samples_leaf=7, min_samples_split=19;, score=0.684 total time=  21.5s
[CV 4/5] END max_depth=22, min_samples_leaf=7, min_samples_split=19;, score=0.694 total time=  21.9s
[CV 5/5] END max_depth=22, min

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....001FE669D55B0>, 'min_samples_leaf': <scipy.stats....001FE66C457F0>, 'min_samples_split': <scipy.stats....001FE669D5810>}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [16]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
2,21,4,17,0.689321
5,22,5,20,0.688288
6,19,4,22,0.687904
3,21,5,25,0.687201
7,20,4,26,0.686618
4,22,7,18,0.686223
8,19,3,26,0.686195
0,21,6,27,0.685718
1,22,7,19,0.685613
9,24,8,27,0.684432


In [17]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 21, 'min_samples_leaf': 4, 'min_samples_split': 17}
Best CV score: 0.6893207197496402


In [18]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      7732
           1       0.98      0.99      0.98      8240

    accuracy                           0.98     15972
   macro avg       0.98      0.98      0.98     15972
weighted avg       0.98      0.98      0.98     15972



#### Evaluación

In [19]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.61      0.55      0.58      1936
           1       0.61      0.67      0.64      2060

    accuracy                           0.61      3996
   macro avg       0.61      0.61      0.61      3996
weighted avg       0.61      0.61      0.61      3996



#### Guardado

In [21]:
os.makedirs(f'./modelos_clasicos/modelos/ventanas/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/ventanas/{dataset}/features_rf.pkl')

['./modelos_clasicos/modelos/ventanas/aug_sin_proc/features_rf.pkl']